# A comparison within the family

**A guided investigation · Chapter 6 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-twins_clean.ipynb)

> **The question**  
> Does the schooling–wage pattern look different when we compare twins?

Comparing unrelated people leaves many possible explanations for a schooling–earnings association. Twins offer a way to hold some family background and inherited characteristics more nearly constant. The Twinsburg research collected information about education and wages, including reports of schooling from both twins. This prepared extract contains individual observations and supplied within-pair differences. It does not give us a general-purpose pair identifier, so we must follow the chapter’s instruction to use records marked first when examining those differences. We will contrast the individual pattern with a chart of schooling and wage differences within pairs, then inspect disagreement between schooling reports. Similar upbringing does not eliminate every difference between twins, and reported schooling can contain error. This case shows both the appeal of a closer comparison and the new questions it introduces.

### By the end, you will be able to…

- Compare individuals with within-pair differences, and see why measurement quality matters.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

What influences on earnings might twins share, and what influences might still differ?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Individual twin within a twin pair.  
**Stored size:** 680 rows × 11 columns.

| Column | What it records |
|---|---|
| `educ` | Own-reported years of education. |
| `lwage` | Natural logarithm of wage in the twins source; not a dollar wage. |
| `first` | Indicator selecting the first record per twin pair; use 1 for supplied pair differences. |
| `deduc` | Supplied within-pair difference in own-reported years of education. |
| `dlwage` | Supplied within-pair difference in log wages. |

Use first == 1 once per pair for supplied differences. Do not infer pairs from row order. lwage and dlwage are log-wage quantities, not dollar wages.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch6/twins_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['educ', 'lwage', 'first', 'deduc', 'dlwage']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['educ', 'lwage', 'first', 'deduc', 'dlwage']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'age': 'Age (years)',
 'census': 'Census year',
 'deduc': 'Own-reported schooling difference (years)',
 'deduct': 'Twin-reported schooling difference (years)',
 'dlwage': 'Within-pair log-wage difference',
 'educ': 'Own-reported schooling (years)',
 'group': 'State group',
 'lwage': 'Log wage',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state'}

## Visual investigation 1 · Start with the individual pattern

**Predict:** Do more-schooled twins tend to have higher wages?

Each dot is one stored individual. Discrete years of schooling produce vertical stacks; transparency helps reveal overlap. Wages are on a logarithmic scale.

In [ ]:
fig = px.scatter(df, x="educ", y="lwage", opacity=0.5, labels=labels, title="Individuals: schooling and log wages")
fig.show()

**Explore the chart:** Hover over observations at the same schooling level. How much variation remains?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The individual association can reflect both schooling and background. A family comparison changes what variation we use.

</details>

## Visual investigation 2 · Switch to within-pair differences

**Predict:** When one twin has more schooling, does that twin also have higher wages?

The chapter uses first == 1 to include supplied differences once per pair. Zero means the twins have the same reported schooling or log wage. The origin is the natural reference point.

In [ ]:
pairs = df[df["first"] == 1].copy()
fig = px.scatter(pairs, x="deduc", y="dlwage", opacity=0.6, labels=labels, title="Within pairs: compare differences")
fig.add_hline(y=0, line_dash="dot")
fig.add_vline(x=0, line_dash="dot")
fig.show()

**Explore the chart:** Find points with zero schooling difference but nonzero wage difference. What does that tell you?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Twins can differ in earnings even when schooling matches. Shared background is only one set of influences, and pair differences can contain measurement error.

</details>

## Visual investigation 3 · Inspect the measurement itself

**Predict:** Do own-reported and twin-reported schooling differences agree?

A point on a 45-degree diagonal would represent agreement between the two reports. Both axes measure a schooling difference in years.

In [ ]:
fig = px.scatter(pairs, x="deduc", y="deduct", opacity=0.6, labels=labels, title="Two reports of the same schooling difference")
limits = [min(pairs["deduc"].min(), pairs["deduct"].min()), max(pairs["deduc"].max(), pairs["deduct"].max())]
fig.add_shape(type="line", x0=limits[0], y0=limits[0], x1=limits[1], y1=limits[1], line=dict(dash="dash", color="#7c3aed"))
fig.show()

**Explore the chart:** Hover away from the agreement line. Could differences in reporting be large relative to differences in schooling?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Disagreement makes measurement error a concrete concern. It does not reveal which report is correct or establish independence of the errors.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Restrict the pair scatter to deduc between −4 and 4. State that this is a restricted view, not the full sample.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Make a histogram of within-pair schooling differences using one record per pair.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Use pairs, not df.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

px.histogram(pairs, x="deduc") displays the distribution of differences.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>fig = px.histogram(pairs, x=&quot;deduc&quot;, labels=labels, title=&quot;How much within-pair schooling variation is available?&quot;)
fig.show()</code></pre>

If many pairs have similar schooling, identifying an effect relies on a smaller portion of the sample. Measurement error can then be especially consequential.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**Why might removing shared background make measurement error in schooling differences more important?**

[Continue with Chapter 6: The Wages of Schooling](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/06-wages-of-schooling.html#strategy-1-twin-comparisons) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch6-twins-clean-csv)

### Source and preparation notes

[Twinsburg studies: Ashenfelter and Krueger (1994); Ashenfelter and Rouse (1998). Book replication resource](https://www.masteringmetrics.com/resources-post/chapter-6-table-6-2/)

Prepared analysis extract from the Twinsburg study; notebook subsets/constructs twin-pair variables. Source preparation details beyond the notebook are not available here.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.